In [0]:
import requests

WORKSPACE_URL = "https://adb-7405615437469430.10.azuredatabricks.net"
TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

payload = {
    "name": "azure-openai-gpt4o",
    "config": {
        "served_entities": [{
            "external_model": {
                "name": "gpt-4o",
                "provider": "openai",
                "task": "llm/v1/chat",
                "openai_config": {
                    "openai_api_type": "azure",
                    "openai_api_key_plaintext": "YOUR_NEW_API_KEY",
                    "openai_api_base": "https://dbx-openai.openai.azure.com",
                    "openai_deployment_name": "gpt-4o",
                    "openai_api_version": "2024-08-01-preview"
                }
            }
        }]
    }
}

response = requests.post(
    f"{WORKSPACE_URL}/api/2.0/serving-endpoints",
    headers={"Authorization": f"Bearer {TOKEN}"},
    json=payload
)

print(response.status_code)
print(response.json())

In [0]:
import openai

client = openai.AzureOpenAI(
    api_key="YOUR_AZURE_OPENAI_KEY",
    api_version="2024-08-01-preview",
    azure_endpoint="https://dbx-openai.openai.azure.com"
)

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": "Hello!"}]
)

print(response.choices[0].message.content)

In [0]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType
import openai

def analyze_review(review_text):
    client = openai.AzureOpenAI(
        api_key="YOUR_AZURE_OPENAI_KEY",
        api_version="2024-08-01-preview",
        azure_endpoint="https://dbx-openai.openai.azure.com"
    )
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": 
            'Analyze the following review and return ONLY a valid JSON object. '
            'Review text: ' + review_text
        }]
    )
    return response.choices[0].message.content

analyze_review_udf = udf(analyze_review, StringType())

# Test it
df = spark.table("ws_dbxproject.`01_bronze`.reviews").limit(3)
df.withColumn("analysis_json", analyze_review_udf(col("review_text"))).show(truncate=False)

In [0]:
%pip install openai